# Day 20 — Mock Interview Practice
**Week 3, Day 6 · Nov 2, 2026 · 18 questions · 8 categories**

**Difficulty: 13 ADVANCED + 5 HARD** — HARD: Q4 (text-to-SQL eval harness), Q8 (capacity planning), Q10 (chaos/DR game day), Q14 (clean rooms), Q16 (LLMOps observability)

| # | Category | Topics |
|---|----------|--------|
| Q1–Q2 | SQL Advanced | Migration reconciliation with EXCEPT ALL · FIFO inventory cost allocation |
| Q3–Q4 | Python | LFU cache in O(1) · Text-to-SQL evaluation harness |
| Q5–Q6 | PySpark Coding | Leakage-safe stratified split · Streaming SCD-dimension enrichment |
| Q7–Q8 | Spark Optimization | CBO & Delta statistics · Platform capacity planning |
| Q9–Q11 | ETL | Exactly-once across systems · Chaos/DR game day · Multi-tenant isolation |
| Q12–Q13 | Data Modeling | UC row filters & column masks · Aggregate facts & aggregate navigation |
| Q14–Q15 | Snowflake | Clean rooms · Horizon governance suite |
| Q16–Q18 | Monitoring, Logging & Orchestration | LLMOps observability · Flink vs Structured Streaming · Reliability analytics |

> Attempt each question BEFORE opening its solution. Score yourself in the score card at the end (1 = could not start, 5 = interview-ready answer).

## Q1 — SQL Advanced: Migration Reconciliation with EXCEPT ALL [ADVANCED]

**Interviewer (Stripe):** We just migrated a 500M-row `orders` table from legacy Oracle to Delta Lake. QA asks: "is the migrated data identical?" A plain `EXCEPT` is not enough — it silently de-duplicates.

**Why EXCEPT fails here:** if the source legitimately contains TWO identical rows, `EXCEPT` cancels them out and reports zero diff. `EXCEPT ALL` preserves row multiplicity — it removes one matching pair at a time, so a row missing in the destination still shows up.

**Your task**, given `src_orders` (Oracle extract) and `dst_orders` (Delta):
1. Rows present in source but missing (or under-counted) in destination
2. Phantom rows present only in the destination
3. For matching keys, exactly WHICH columns mismatch (drill-down report)

**Hint:** two `EXCEPT ALL` passes (one per direction), then a key-join for the column drill-down.

In [0]:
# Solution Q1 — Migration reconciliation with EXCEPT ALL
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

# Oracle extract: pk 1 appears TWICE (legitimate duplicate) — EXCEPT would hide it
src = spark.createDataFrame([(1,'A',100.0),(1,'A',100.0),(2,'B',250.0),(3,'C',75.5),(4,'D',90.0)], 'order_id INT, region STRING, amount DOUBLE')
dst = spark.createDataFrame([(1,'A',100.0),(1,'A',100.0),(2,'B',999.0),(4,'D',90.0),(5,'Z',50.0)], 'order_id INT, region STRING, amount DOUBLE')
src.createOrReplaceTempView('src_orders')
dst.createOrReplaceTempView('dst_orders')

# --- Pass 1 & 2: dup-aware full-duplex diff (EXCEPT ALL keeps multiplicity) ---
missing_in_dst = spark.sql('SELECT * FROM src_orders EXCEPT ALL SELECT * FROM dst_orders')   # order 3 lost + order 2's real row
phantom_in_dst = spark.sql('SELECT * FROM dst_orders EXCEPT ALL SELECT * FROM src_orders')   # order 2's bad row + order 5
n_missing, n_phantom = missing_in_dst.count(), phantom_in_dst.count()
print('missing/under-counted in dest:', n_missing, '| phantom rows in dest:', n_phantom)
assert n_missing == 2 and n_phantom == 2

# --- Pass 3: column-level drill-down on matching keys ---
col_drill = spark.sql('''
SELECT s.order_id,
       CASE WHEN s.region != d.region THEN 'region' END AS region_diff,
       CASE WHEN s.amount != d.amount THEN 'amount' END AS amount_diff
FROM (SELECT DISTINCT order_id, region, amount FROM src_orders) s
JOIN (SELECT DISTINCT order_id, region, amount FROM dst_orders) d USING (order_id)
WHERE s.region != d.region OR s.amount != d.amount
''').collect()
bad_ids = sorted(r['order_id'] for r in col_drill)
print('column-level mismatched keys:', bad_ids, '-> amount column for order 2 (250 vs 999)')
assert bad_ids == [2]
print('VERDICT: 1 row lost (order 3), 1 phantom (order 5), 1 amount corruption (order 2) — dup rows for order 1 reconciled exactly.')

## Q2 — SQL Advanced: FIFO Inventory Cost Allocation with Window Frames [ADVANCED]

**Interviewer (Amazon):** Inventory receipts arrive in batches with unit costs; sales consume the OLDEST batch first (FIFO). Write pure SQL — window functions only, no UDFs, no loops — to compute per-sale COGS and the remaining stock per batch.

**Data:**
* Receipts: R1 = 100 units @ 10.0, R2 = 100 @ 12.0, R3 = 100 @ 15.0
* Sales: S1 = 150 units, S2 = 120 units

**Expected:** S1 COGS = 100×10 + 50×12 = **1600**; S2 COGS = 50×12 + 70×15 = **1650**; leftover = 30 units @ 15.0.

**Hint:** running totals on BOTH sides. Units of receipt r consumed by sale s = `max(0, min(cum_r, cum_s) − max(prev_r, prev_s))` — the overlap of two cumulative intervals.

In [0]:
# Solution Q2 — FIFO allocation via cumulative windows (interval-overlap trick)
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

receipts = spark.createDataFrame([('R1','2026-01-01',100,10.0),('R2','2026-01-05',100,12.0),('R3','2026-01-10',100,15.0)], 'batch STRING, recv_date STRING, qty INT, unit_cost DOUBLE')
sales = spark.createDataFrame([('S1','2026-01-03',150),('S2','2026-01-12',120)], 'sale_id STRING, sale_date STRING, qty INT')
receipts.createOrReplaceTempView('receipts')
sales.createOrReplaceTempView('sales')

# cum_qty = running total; prev_qty = cum_qty - qty  (the interval covered by each batch/sale)
alloc = spark.sql('''
WITH r AS (
  SELECT batch, qty, unit_cost,
         SUM(qty) OVER (ORDER BY recv_date, batch) - qty AS prev_qty,
         SUM(qty) OVER (ORDER BY recv_date, batch)         AS cum_qty
  FROM receipts
), s AS (
  SELECT sale_id, qty,
         SUM(qty) OVER (ORDER BY sale_id) - qty AS prev_qty,
         SUM(qty) OVER (ORDER BY sale_id)         AS cum_qty
  FROM sales
)
SELECT s.sale_id, r.batch, r.unit_cost,
       GREATEST(0, LEAST(r.cum_qty, s.cum_qty) - GREATEST(r.prev_qty, s.prev_qty)) AS alloc_qty
FROM s CROSS JOIN r
''')
alloc.createOrReplaceTempView('alloc')

cogs = spark.sql('SELECT sale_id, SUM(alloc_qty * unit_cost) AS cogs FROM alloc WHERE alloc_qty > 0 GROUP BY sale_id ORDER BY sale_id')
rows = {r['sale_id']: r['cogs'] for r in cogs.collect()}
print('per-sale COGS:', rows)
assert abs(rows['S1'] - 1600.0) < 1e-6 and abs(rows['S2'] - 1650.0) < 1e-6

leftover = spark.sql('''
SELECT batch, unit_cost,
       cum_qty - (SELECT COALESCE(SUM(alloc_qty),0) FROM alloc) AS remaining
FROM (SELECT batch, unit_cost, SUM(qty) OVER (ORDER BY recv_date, batch) AS cum_qty FROM receipts)
ORDER BY batch
''').collect()
for r in leftover: print(r)
last = leftover[-1]
assert last['batch'] == 'R3' and last['remaining'] == 30
print('FIFO verified: S1=1600, S2=1650, leftover 30 units @ 15.0 (R1/R2 fully consumed).')

## Q3 — Python: LFU Cache in O(1) [ADVANCED]

**Interviewer (Uber):** Design an LFU (Least Frequently USED) cache with `get` and `put` in **O(1)**. Eviction order: least frequently used first; ties broken by least recently used.

**Ops sequence (LeetCode 460), capacity 2:**
`put(1,1)`, `put(2,2)`, `get(1)`, `put(3,3)`, `get(2)`, `get(3)`, `put(4,4)`, `get(1)`, `get(3)`, `get(4)`
→ key **2** evicted first (freq 1), then key **1** (freq tie 2:2, LRU loses).

**Be ready to answer:**
1. Why `OrderedDict` buckets give O(1) LRU-within-frequency
2. Why you track `min_freq` (and when it bumps up)
3. What changes for LRU-K / weighted frequency / TTL expiry

**Hint:** three maps: key → (value, freq), freq → OrderedDict of keys, plus min_freq.

In [0]:
# Solution Q3 — LFU cache: freq buckets of OrderedDicts (all ops O(1))
from collections import OrderedDict

class LFUCache:
    def __init__(self, capacity):
        self.cap = capacity
        self.size = 0
        self.min_freq = 0
        self.keymap = {}    # key -> (freq, value)
        self.freqs = {}     # freq -> OrderedDict of keys (insertion order = LRU order)

    def _touch(self, key):
        f, val = self.keymap[key]
        self.freqs[f].pop(key)                      # O(1) — pop from anywhere in OrderedDict
        if not self.freqs[f]:
            del self.freqs[f]
            if f == self.min_freq:                   # bucket empty -> min_freq moves up
                self.min_freq = f + 1
        self.freqs.setdefault(f + 1, OrderedDict())[key] = None
        self.keymap[key] = (f + 1, val)

    def get(self, key):
        if key not in self.keymap: return -1
        self._touch(key)
        return self.keymap[key][1]

    def put(self, key, value):
        if self.cap <= 0: return
        if key in self.keymap:                       # update + promote
            self.keymap[key] = (self.keymap[key][0], value)
            self._touch(key)
            return
        if self.size == self.cap:                    # evict LRU within the LFU bucket
            evict, _ = self.freqs[self.min_freq].popitem(last=False)
            del self.keymap[evict]
            self.size -= 1
        self.min_freq = 1                            # a fresh key starts at freq 1
        self.freqs.setdefault(1, OrderedDict())[key] = None
        self.keymap[key] = (1, value)
        self.size += 1

c = LFUCache(2)
c.put(1, 1); c.put(2, 2)
assert c.get(1) == 1
print('key 1 freq -> 2; key 2 stays freq 1')
c.put(3, 3)                       # evicts key 2 (min freq)
assert c.get(2) == -1
assert c.get(3) == 3
c.put(4, 4)                       # keys 1 and 3 both freq 2 -> evict LRU = key 1
assert c.get(1) == -1
assert c.get(3) == 3 and c.get(4) == 4
print('LFU verified: evictions in order 2 (freq), then 1 (LRU tie-break) — every op O(1).')

# Interview extensions: LRU-K = track last K uses; weighted freq = decay old counts;
# TTL = per-key expiry heap or lazy check on get. All keep O(1) amortized with the bucket trick.

## Q4 — Python: Text-to-SQL Evaluation Harness [HARD] *(2025–2026 trending)*

**Interviewer (Meta):** We're rolling out a text-to-SQL agent to 500 analysts. Before every release you must answer: did it get better or worse? Comparing SQL strings is useless — the same question has many valid spellings.

**Build an eval harness with:**
1. A **golden set**: question → gold SQL (curated, version-controlled)
2. **Execution match** — run gold vs candidate on real tables and compare RESULT SETS (alias/format-insensitive, order-normalized)
3. **Rubric partial credit** for near-misses: correct tables (+0.3), aggregation present (+0.3), filters/grouping (+0.2), shape (+0.2)
4. Composite score (70% execution match + 30% rubric) + a **regression gate** that blocks the release if score < previous release

**Must include a case where execution-match says CORRECT but string diff would say WRONG** (e.g. `COUNT(*)` vs `COUNT(order_id)`).

**Talk track:** where this runs in CI, how you grow the golden set from production queries, and why eval-in-CI is the LLM equivalent of unit tests.

In [0]:
# Solution Q4 — text-to-SQL eval harness: execution match + rubric + regression gate
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sql('CREATE OR REPLACE TEMP VIEW orders AS SELECT * FROM VALUES (1, "a", 100.0), (2, "a", 250.0), (3, "b", 75.0), (4, "b", 90.0) AS t(order_id, region, amount)')

GOLDEN = [
    ('total revenue by region', 'SELECT region, SUM(amount) AS rev FROM orders GROUP BY region ORDER BY region'),
    ('orders over 80',          'SELECT order_id FROM orders WHERE amount > 80 ORDER BY order_id'),
    ('count of orders',         'SELECT COUNT(*) AS n FROM orders'),
    ('top region by revenue',   'SELECT region, SUM(amount) AS rev FROM orders GROUP BY region ORDER BY rev DESC LIMIT 1'),
]
CANDIDATES = [
    'SELECT region, SUM(amount) AS total_rev FROM orders GROUP BY region ORDER BY region',      # correct, different alias
    'SELECT order_id FROM orders WHERE amount > 80 AND amount < 200 ORDER BY order_id',           # wrong: drops the 250 order
    'SELECT COUNT(order_id) AS cnt FROM orders',                                                 # CORRECT results, different spelling
    'SELECT region, MAX(amount) AS rev FROM orders GROUP BY region ORDER BY rev DESC LIMIT 1',     # wrong: MAX instead of SUM
]

def exec_match(spark, gold_sql, cand_sql):
    try:
        g = [tuple(r) for r in spark.sql(gold_sql).collect()]
        c = [tuple(r) for r in spark.sql(cand_sql).collect()]
        return sorted(g) == sorted(c)          # order-insensitive result-set equality
    except Exception:
        return False

RUBRIC = {'tables': 0.3, 'aggregation': 0.3, 'filter': 0.2, 'shape': 0.2}
def partial_credit(cand_sql):
    score = 0.0
    if 'orders' in cand_sql: score += RUBRIC['tables']
    if any(a in cand_sql for a in ('SUM(', 'COUNT(', 'MAX(', 'AVG(')): score += RUBRIC['aggregation']
    if 'WHERE' in cand_sql or 'GROUP BY' in cand_sql: score += RUBRIC['filter']
    if cand_sql.rstrip().rstrip(';').upper().startswith('SELECT'): score += RUBRIC['shape']
    return score

em = [exec_match(spark, g, c) for (_, g), c in zip(GOLDEN, CANDIDATES)]
pc = [partial_credit(c) for c in CANDIDATES]
overall = sum(0.7 * e + 0.3 * p for e, p in zip(em, pc)) / len(GOLDEN)
print('execution matches:', em)
print('case 3 proves the point: COUNT(*) vs COUNT(order_id) -> different string, SAME result = CORRECT')
print('overall score:', round(overall, 3))
assert em == [True, False, True, False]

PREV_RELEASE = 0.85
gate_pass = overall >= PREV_RELEASE
print(f'regression gate (>= {PREV_RELEASE}):', 'PASS' if gate_pass else 'FAIL — BLOCK RELEASE')
assert not gate_pass
print('Harness verdict: 50% accuracy, 0.612 composite — gate blocks this release until the query planner or prompts improve.')

## Q5 — PySpark: Leakage-Safe Stratified Train/Test Split [ADVANCED]

**Interviewer (Netflix):** Churn model, 1000 customers, 90/10 class imbalance. Two traps to avoid:
1. `randomSplit` is **re-run non-deterministic** (different split each run) and can under-sample churners
2. Building train and test sets with two independent sampling passes can leak the SAME customer into both

**Your task:**
1. Use `sampleBy(col, fractions, seed)` to oversample the minority class in a stratified sample
2. Build the real split deterministically: `pmod(hash(cust_id), 100) < 70` → train, else test
3. **Prove all three properties:** zero train/test overlap, churn rate balanced across splits (within 5pp), and the split is stable on re-run (idempotent)

**Talk track:** why hashing beats randomSplit (reproducibility, distributed re-splitting, incremental new rows get a stable home), and when time-based splits matter more (churn with concept drift).

In [0]:
# Solution Q5 — sampleBy for stratified sampling + deterministic hash split
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

rows = [(f'c{i}', 1 if i >= 900 else 0) for i in range(1000)]      # 900 non-churners, 100 churners
churn = spark.createDataFrame(rows, 'cust_id STRING, churn INT')
churn.createOrReplaceTempView('churn')

# 1) sampleBy: stratified sampling with per-class fractions — oversample the 10% class
strat = churn.sampleBy('churn', {0: 0.2, 1: 0.5}, seed=7)
strat_counts = strat.groupBy('churn').count().orderBy('churn').collect()
print('sampleBy sample (churn=0, churn=1):', [(r['churn'], r['count']) for r in strat_counts])
assert strat_counts[0]['count'] >= 150 and strat_counts[1]['count'] >= 40

# 2) deterministic hash split — same customer lands in the SAME side on every run
split = spark.sql('''
SELECT cust_id, churn,
       CASE WHEN pmod(hash(cust_id), 100) < 70 THEN 'train' ELSE 'test' END AS split
FROM churn
''')
split.createOrReplaceTempView('churn_split')

# 3a) leakage check: no cust_id on both sides
leak = spark.sql('SELECT COUNT(*) AS n FROM (SELECT cust_id, COUNT(DISTINCT split) ns FROM churn_split GROUP BY cust_id) WHERE ns > 1').collect()[0]['n']
assert leak == 0
# 3b) balance check: churn rate within 5 percentage points across sides
prop = spark.sql('SELECT split, ROUND(AVG(churn), 3) AS churn_rate, COUNT(*) n FROM churn_split GROUP BY split ORDER BY split').collect()
for r in prop: print(r)
rates = [r['churn_rate'] for r in prop]
assert abs(rates[0] - rates[1]) < 0.05
print('PROVEN: 0 overlapping customers, churn rate ~10% on both sides, deterministic via pmod(hash).')

## Q6 — PySpark: Streaming SCD-Dimension Enrichment (as-of Event Time) [ADVANCED]

**Interviewer (DoorDash):** A fact event stream needs `customer_segment`, but the dimension is SCD Type 2 — customer 100 was STANDARD until Jan 10, then PREMIUM. Joining the stream to the CURRENT dim row mislabels every historical event.

**Your task:**
1. Stream events from parquet using `foreachBatch` + `trigger(availableNow=True)` (the batch-on-demand streaming pattern)
2. Enrich each micro-batch with an **as-of join**: `event_ts >= valid_from AND event_ts < valid_to`
3. **Prove the naive current-row join gives the WRONG segment** for the Jan 5 event

**Be ready to explain:** why stream-static joins re-read the dim per batch (and when to cache it), how dim updates eventually get picked up, and when you'd rebuild history instead (backfill with full outer + restate).

In [0]:
# Solution Q6 — foreachBatch + as-of join against an SCD2 dimension
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
spark = SparkSession.builder.getOrCreate()

BASE = '/Volumes/workspace/default/etl_test_volume/d20_q6'
# Re-run-safe WITHOUT destructive cleanup: overwritten parquet files get NEW timestamps,
# so the checkpoint re-ingests them, and the enriched table is overwritten (not appended).

# SCD Type 2 dimension: customer 100 switches segment on Jan 10
dim = spark.createDataFrame([
    (100, 'STANDARD', '2026-01-01 00:00:00', '2026-01-10 00:00:00'),
    (100, 'PREMIUM',   '2026-01-10 00:00:00', '9999-12-31 23:59:59'),
    (200, 'STANDARD', '2026-01-01 00:00:00', '9999-12-31 23:59:59'),
], 'cust_id INT, segment STRING, valid_from STRING, valid_to STRING')
dim.createOrReplaceTempView('d20_dim')

events = spark.createDataFrame([
    (1, '2026-01-05 12:00:00', 100, 40.0),   # BEFORE switch -> STANDARD
    (2, '2026-01-12 12:00:00', 100, 55.0),   # AFTER switch  -> PREMIUM
    (3, '2026-01-05 12:00:00', 200, 10.0),
], 'event_id INT, event_ts STRING, cust_id INT, amount DOUBLE')
events.write.mode('overwrite').parquet(f'{BASE}/events')

def enrich(batch_df, batch_id):
    b = batch_df.withColumn('ts', F.to_timestamp('event_ts'))
    d = spark.table('d20_dim').withColumn('vf', F.to_timestamp('valid_from')).withColumn('vt', F.to_timestamp('valid_to'))
    enriched = (b.alias('e')
                .join(d.alias('d'), (F.col('e.cust_id') == F.col('d.cust_id')) &
                                    (F.col('e.ts') >= F.col('d.vf')) & (F.col('e.ts') < F.col('d.vt')), 'left')
                .select('e.event_id', 'e.event_ts', 'e.cust_id', 'e.amount', 'd.segment'))
    enriched.write.format('delta').mode('overwrite').save(f'{BASE}/enriched')   # single-batch demo: overwrite = re-runnable

schema = StructType([
    StructField('event_id', IntegerType()), StructField('event_ts', StringType()),
    StructField('cust_id', IntegerType()), StructField('amount', DoubleType())])
q = (spark.readStream.format('parquet').schema(schema).load(f'{BASE}/events')   # explicit schema: streams don't infer
     .writeStream.foreachBatch(enrich)
     .option('checkpointLocation', f'{BASE}/ckpt')
     .trigger(availableNow=True)                                                  # run-to-completion, then stop
     .start())
q.awaitTermination(60)

res = spark.read.format('delta').load(f'{BASE}/enriched').orderBy('event_id').collect()
for r in res: print(r)
segs = {r['event_id']: r['segment'] for r in res}
assert segs[1] == 'STANDARD' and segs[2] == 'PREMIUM' and segs[3] == 'STANDARD'

# The naive CURRENT-row join mislabels the Jan 5 event as PREMIUM — classic silent corruption
naive = spark.sql(f'SELECT e.event_id, d.segment FROM parquet.`{BASE}/events` e JOIN d20_dim d ON e.cust_id = d.cust_id AND d.valid_to = \'9999-12-31 23:59:59\'')
naive_rows = {r['event_id']: r['segment'] for r in naive.collect()}
print('naive current-only join:', naive_rows, '<- event 1 WRONGLY labeled PREMIUM')
assert naive_rows[1] == 'PREMIUM'
print('As-of join verified: event-time correctness that a current-row join cannot give.')

## Q7 — Spark Optimization: CBO & Delta Statistics [ADVANCED]

**Interviewer (Databricks):** Spark's planner long relied on `sizeInBytes` heuristics — no cardinalities, no NDV. Delta changed this. Explain the cost-based optimizer story for the Lakehouse and demonstrate it.

**Answer with code:**
1. `ANALYZE TABLE ... COMPUTE DELTA STATISTICS` — gathers table-level stats (rows, bytes) at commit 0 AND column-level stats (min/max, NDV, null counts) in the background
2. Show the stats via `DESCRIBE EXTENDED`
3. Show the planner choosing `BroadcastHashJoin` BASED on those stats
4. **Explain the operational side:** stats go stale after MERGE/overwrite; first statistics gather is triggered at ~10GB or 1000 files; column stats drive join estimation, range-join hints (`RANGE_JOIN`), and filter selectivity

In [0]:
# Solution Q7 — ANALYZE / Delta statistics + the join decision they drive
from pyspark.sql import SparkSession
import io, contextlib
spark = SparkSession.builder.getOrCreate()

# fresh demo tables (CREATE OR REPLACE keeps this idempotent, no DROP needed)
spark.sql('CREATE OR REPLACE TABLE workspace.default.d20_q7_fact AS SELECT id AS order_id, CAST(id % 7 AS INT) AS region_id, (id * 13.5) AS amount FROM range(20000)')
spark.sql('CREATE OR REPLACE TABLE workspace.default.d20_q7_dim AS SELECT id AS region_id, CONCAT(\'REGION_\', id) AS region_name FROM range(7)')

# gather statistics for both tables
spark.sql('ANALYZE TABLE workspace.default.d20_q7_fact COMPUTE DELTA STATISTICS')
spark.sql('ANALYZE TABLE workspace.default.d20_q7_dim COMPUTE DELTA STATISTICS')
print('ANALYZE completed.')

for r in spark.sql('DESCRIBE EXTENDED workspace.default.d20_q7_fact').collect():
    if 'Statistics' in str(r['col_name']):
        print('fact stats:', r['col_name'], '=>', str(r['data_type'])[:80])

plan = spark.sql('SELECT f.order_id, d.region_name, f.amount FROM workspace.default.d20_q7_fact f JOIN workspace.default.d20_q7_dim d ON f.region_id = d.region_id')
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    plan.explain()
p = buf.getvalue()
first_lines = [l for l in p.splitlines() if 'Join' in l][:2]
print('join in physical plan:', first_lines)
assert 'Broadcast' in p
print('Planner chose BroadcastHashJoin — 7-row dim + fresh stats -> broadcast. Without ANALYZE, a big fact\'s unknown size can push the planner to a shuffled join (or worse, a broadcast of a table that does not fit).')
print('Operational notes: (1) rerun ANALYZE after heavy MERGE/OVERWRITE — stats do not auto-refresh mid-table; (2) Delta gathers the FIRST statistics at commit 0, then periodically (~10GB / 1000 files); (3) column stats (NDV, min/max) feed join cardinality estimates and RANGE_JOIN optimization.')

## Q8 — Spark Optimization: Platform Capacity Planning [HARD]

**Interviewer (Amazon):** 20 teams share one lakehouse platform. Nightly workload: bronze ingest 40 task-hours, silver transform 90, gold serve 25, ML training 80. Batch window 4h, 8 task slots per worker, peak concurrency factor 1.6×. Spot workers cost 0.7× on-demand DBUs, but the reliability SLO caps spot share at 60%. Monthly budget: 12 worker-units.

**Compute:**
1. Peak task-hours and the minimum worker count that fits the window
2. The cost-optimal spot/on-demand mix UNDER the 60% SLO cap
3. Whether you fit the budget — and what breaks first at 2× growth

**Talk track:** where autoscaling changes the math (workers ≈ f(arrival rate × task time / slots), not fixed), why queue-wait SLOs ("no job waits > 15 min") set the true floor, and the spot-diversity trick (multiple instance types reduce spot-reclaim collisions).

In [0]:
# Solution Q8 — capacity model: task-hours -> workers -> optimal mix -> budget
import math

workload = {'bronze_ingest': 40, 'silver_transform': 90, 'gold_serve': 25, 'ml_training': 80}   # task-hours/night
PEAK_FACTOR = 1.6      # observed peak-concurrency multiplier
window_hours = 4       # nightly batch window
slots_per_worker = 8   # parallel task slots per worker
COST = {'spot': 0.7, 'ondemand': 1.0}     # DBU multipliers
SLO_SPOT_CAP = 0.60    # reliability SLO: at most 60% spot
budget = 12.0          # worker-units per month

# 1) peak load -> worker count
peak_task_hours = sum(workload.values()) * PEAK_FACTOR
workers = math.ceil(peak_task_hours / (window_hours * slots_per_worker))
print(f'total task-hours {sum(workload.values())} × peak {PEAK_FACTOR} = {peak_task_hours} peak task-hours')
print(f'-> {workers} workers needed to drain the window in {window_hours}h at {slots_per_worker} slots each')
assert workers == 12

# 2) cost-optimal mix under the SLO cap (brute-force over spot share — the LP is tiny)
best = None
for spot_pct in range(0, 101, 5):
    if spot_pct / 100 > SLO_SPOT_CAP:      # SLO constraint binds
        continue
    cost = workers * (spot_pct / 100 * COST['spot'] + (1 - spot_pct / 100) * COST['ondemand'])
    if best is None or cost < best[1]:
        best = (spot_pct, cost)
print(f'optimal mix: {best[0]}% spot / {100 - best[0]}% on-demand at {best[1]:.1f} worker-units (SLO binds at the cap)')
assert best[0] == 60 and abs(best[1] - 12 * 0.82) < 1e-9

# 3) budget check + 2x growth stress
print(f'budget check: {best[1]:.1f} <= {budget} -> fits, but only ~18% headroom — tight already')
assert best[1] <= budget
growth_workers = math.ceil(peak_task_hours * 2 / (window_hours * slots_per_worker))
growth_cost = growth_workers * (0.6 * 0.7 + 0.4 * 1.0)
print(f'at 2x growth: {growth_workers} workers needed, {growth_cost:.1f} worker-units — ', 'still fits' if growth_cost <= budget else 'EXCEEDS budget -> time to right-size workloads or renegotiate')
assert growth_cost > budget
print('Conclusion: fine today, breaks at ~2x. Queue SLO + autoscaling (not fixed workers) is the answer to bursty multi-team load; spot diversity keeps reclaim probability low at 60%.')

## Q9 — ETL: Exactly-Once ACROSS Systems [ADVANCED]

**Interviewer (Confluent-style):** Pipeline: Kafka → Spark Structured Streaming → Delta. Spark guarantees exactly-once for the Spark→Delta hop (transactional writes + idempotent sinks). The DANGER WINDOW is BETWEEN systems: rows already written to Delta but Kafka offsets not yet committed. Crash there → replay re-reads offsets and writes duplicates.

**Your task (simulate end-to-end):**
1. Show the naive pattern (offsets in a separate system, committed AFTER writes) producing **4 duplicates** after a crash + replay
2. The **offset-journal pattern**: store source offsets in the SAME Delta transaction as the rows — replay resumes from the journal, no duplicates, no loss
3. **Epoch fencing**: why a zombie writer from the old generation must be rejected

*(Distinct from Day 8/11: those covered exactly-once WITHIN Spark's sink semantics — this is the cross-system end-to-end guarantee.)*

In [0]:
# Solution Q9 — cross-system exactly-once: offset journal + epoch fencing (simulated)

class KafkaLog:
    def __init__(self, msgs): self.log = msgs                      # [(partition, offset, payload)]
    def read(self, from_offsets):                                   # {partition: next_offset}
        return [(p, o, m) for (p, o, m) in self.log if o >= from_offsets.get(p, 0)]

class DeltaSink:
    def __init__(self):
        self.rows = []
        self.committed_offsets = {}
    def commit(self, batch_rows, offsets):
        # THE KEY IDEA: rows + source offsets land in ONE atomic Delta transaction
        self.rows += batch_rows
        self.committed_offsets = offsets

kafka = KafkaLog([(0, 0, 'a'), (0, 1, 'b'), (1, 0, 'c'), (1, 1, 'd')])

# ---- 1) NAIVE: offsets in a separate system, committed AFTER the rows ----
naive_rows, committed_offsets = list(kafka.read({})), {}     # rows written...
# ...CRASH here, before the external offset commit happened
replay = list(kafka.read(committed_offsets))                 # restart re-reads EVERYTHING
dups = len(naive_rows) + len(replay) - 4
print('naive (separate offset commit): rows after crash+replay =', 4 + dups)
assert dups == 4

# ---- 2) CORRECT: offset journal inside the sink transaction ----
sink = DeltaSink()
batch = kafka.read({})
sink.commit([m for (_, _, m) in batch], {0: 2, 1: 2})        # rows + offsets atomically
batch2 = kafka.read(sink.committed_offsets)                  # replay resumes FROM THE JOURNAL
sink.commit([m for (_, _, m) in batch2], {0: 2, 1: 2})
print('journal pattern: rows =', sink.rows, '| replay re-read =', batch2)
assert sink.rows == ['a', 'b', 'c', 'd'] and batch2 == []

# ---- 3) epoch fencing rejects zombie writers after failover ----
active_writer_epoch = 2
zombie_epoch = 1
assert zombie_epoch < active_writer_epoch
print('epoch fencing: zombie (epoch 1) fenced by active writer (epoch 2) — its late commit is rejected, not merged.')
print('Summary: sink-transactional offsets kill the crash window; fencing kills split-brain. Together: end-to-end exactly-once.')

## Q10 — ETL: Chaos / DR Game Day Design [HARD]

**Interviewer (Netflix):** "Our DR plan is a wiki page nobody has read since it was written." Design a recurring chaos/game-day program for the lakehouse that PROVES recovery instead of assuming it.

**Design must cover:**
1. **Scenario matrix** — inject real failures on a staging shadow of production: executor kill mid-shuffle, checkpoint directory deletion, upstream schema change, cloud zone outage
2. Per scenario: the **detection signal** (what alerts first), **RTO/RPO objectives**, and measured recovery times
3. A drill **readiness score** + explicit handling of breaches (fix now vs accept-risk with an alert)
4. Why game days beat documentation, and how to run them safely (blast-radius caps, stakeholder comms, fixed stop conditions)

In [0]:
# Solution Q10 — game-day drill: inject, measure, score, act on breaches

scenarios = [
    # name, detection signal, RTO objective (min), measured RTO (min), RPO objective (min), measured RPO (min)
    ('executor killed mid-shuffle', 'job retry / SPOT_INSTANCE_RECOVERY',   30,  12,  5,  0),
    ('checkpoint directory deleted', 'STREAM_FAILED / DIFFERENT_CHECKPOINT', 60,  55, 15, 10),
    ('upstream schema change',       'data contract validator BLOCKING',     120, 150, 60, 60),
    ('cloud zone outage',            'cluster launch failures',               90,  95, 30, 25),
]

readiness, breaches = [], []
for name, signal, rto_obj, rto, rpo_obj, rpo in scenarios:
    ok = rto <= rto_obj and rpo <= rpo_obj
    readiness.append((name, rto, rto_obj, rpo, rpo_obj, 'PASS' if ok else 'FAIL'))
    print(f'{name:32s} detect: {signal[:34]:34s} RTO {rto:>3}/{rto_obj:<3} RPO {rpo:>2}/{rpo_obj:<2} -> {"PASS" if ok else "FAIL"}')
    if not ok:
        breaches.append(name)

score = (len(scenarios) - len(breaches)) / len(scenarios)
print(f'\ndrill readiness score: {score:.0%} | breaches: {breaches}')
assert len(breaches) == 2 and score == 0.5

# action rules per breach
actions = {
    'upstream schema change': 'FIX NOW — auto-quarantine + runbook v2; 30 min over RTO each drill is trend, not noise',
    'cloud zone outage': 'ACCEPT-RISK + alert — 5 min over RTO once; re-measure next drill before re-architecting',
}
for b in breaches:
    print(f'  {b} -> {actions[b]}')
print('\nWhy game days beat documentation: runbooks decay, muscle memory does not. Caps: staging shadow only, max 1 concurrent injection, auto-abort on customer-facing signal, comms channel open before the first kill.')

## Q11 — ETL: Multi-Tenant Isolation in a Shared Lakehouse [ADVANCED]

**Interviewer (Salesforce):** One shared Delta lakehouse serves 7 tenants with weekly task-hours 4000, 2500, 1200, 800, 300, 150, 50. Cheaper to share — but isolation is now an engineering problem, not a deployment choice.

**Your task:**
1. **Quantify noisy-neighbor risk:** per-tenant compute share + HHI concentration index
2. **Isolation ladder:** schema-per-tenant vs shared-table + row filters vs cell-based — pick per tenant tier (cost, governance, query performance)
3. **Compaction skew:** what tenant t0 (44% of compute) does to OPTIMIZE concurrency, and the fixes (tenant-tiered maintenance windows, dedicated pipelines for whales, per-tenant file layout/liquid clustering)

In [0]:
# Solution Q11 — noisy-neighbor quantification + the isolation ladder

tenant_rows = [(f't{i}', n) for i, n in enumerate([4000, 2500, 1200, 800, 300, 150, 50])]
total = sum(n for _, n in tenant_rows)
shares = [(t, n, n / total) for t, n in tenant_rows]

# 1) concentration: per-tenant share + Herfindahl–Hirschman Index
noisy = [t for t, n, s in shares if s > 0.30]
hh_index = sum(s ** 2 for _, _, s in shares)
print('tenant shares:', [(t, f'{s:.1%}') for t, _, s in shares])
print('noisy neighbors (>30% of compute):', noisy, '| HHI:', round(hh_index, 3), '(0 = perfectly spread, 1 = single tenant)')
assert noisy == ['t0'] and 0.30 < hh_index < 0.31

# 2) isolation ladder — pick per tier
LADDER = [
    ('schema-per-tenant', 'strongest isolation + per-tenant SLAs; N schemas to govern, cross-tenant analytics painful'),
    ('shared table + row filter', 'cheap, one governance surface; filters become join keys — keep them on clustered columns'),
    ('cell-based / dedicated compute', 'whale tenants: own pipelines + maintenance windows, no shared OPTIMIZE queue'),
]
for tier, tradeoff in LADDER:
    print(f'  {tier:28s} {tradeoff}')

recommendation = {t: ('cell-based / dedicated' if s > 0.30 else 'shared + row filter' if s > 0.02 else 'schema-per-tenant (if isolation demanded)' if s > 0.01 else 'shared + row filter') for t, n, s in shares}
print('tiering:', {t: r for t, r in recommendation.items() if r.startswith('cell')})
assert recommendation['t0'] == 'cell-based / dedicated'
print('Compaction skew: t0\'s 44% share means shared OPTIMIZE competes with its writes — tiered maintenance windows + liquid-cluster CLUSTER BY (tenant_id) keep the whale from starving the minnows.')

## Q12 — Data Modeling: Policy-Aware Models with UC Row Filters & Column Masks [ADVANCED]

**Interviewer (Databricks):** Design the gold `sales` model so tenant isolation is enforced by POLICY, not by every analyst remembering a `WHERE tenant = 'acme'` clause — and PII is masked at read time.

**Cover:**
1. The Unity Catalog SQL: `CREATE FUNCTION tenant_only(tenant) RETURN is_account_group_member('acme_analysts') AND tenant = 'acme'` + `ALTER TABLE ... SET ROW FILTER tenant_only ON (tenant)`; column masks via `ALTER TABLE ... ALTER COLUMN email SET MASK email_mask`
2. Simulate the semantics: row-filtered + masked result for an `acme_analysts` group member
3. **Where filters belong in the model:** facts vs dimensions — and the performance rule: a row filter column becomes part of EVERY scan; keep it the join/clustered key
4. How governed tags make this auditable at platform scale (tag-driven policy assignment, not per-table toil)

*(Snowflake RAP/Dynamic Masking was Day 16/9 — this is the Unity Catalog implementation + modeling judgment.)*

In [0]:
# Solution Q12 — row filter + column mask semantics (demo) + the real DDL
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.getOrCreate()

sales = spark.createDataFrame(
    [(1, 'acme', 'ops@acme.com', 100.0), (2, 'globex', 'ceo@globex.com', 250.0)],
    'id INT, tenant STRING, email STRING, amount DOUBLE')

# ---- simulate the SQL policy engine (the real UC syntax is in DDL below) ----
def is_account_group_member(group):
    return group == 'acme_analysts'                    # imagine the session principal check

def apply_row_filter(df, tenant_col, principal_group):
    if is_account_group_member(principal_group):
        return df.where(F.col(tenant_col) == 'acme')   # filter body: tenant = 'acme'
    return df.where(F.lit(False))                      # deny-by-default

def apply_column_mask(df, col):
    return df.withColumn(col, F.regexp_replace(F.col(col), '^.+@', '***@'))

result = apply_column_mask(apply_row_filter(sales, 'tenant', 'acme_analysts'), 'email').collect()
for r in result: print(r)
assert len(result) == 1 and result[0]['email'] == '***@acme.com'
print('acme_analysts sees 1 row (own tenant) with email masked; globex row invisible — policy, not discipline.')

DDL = (
    'CREATE OR REPLACE FUNCTION main.security.tenant_only(tenant STRING)\n'
    'RETURN is_account_group_member(\'acme_analysts\') AND tenant = \'acme\';\n'
    'ALTER TABLE main.sales.silver SET ROW FILTER main.security.tenant_only ON (tenant);\n'
    'CREATE OR REPLACE FUNCTION main.security.mask_email(email STRING)\n'
    'RETURN CASE WHEN is_account_group_member(\'pii_readers\') THEN email ELSE concat(\'***@\', split(email, \'@\')[1]) END;\n'
    'ALTER TABLE main.sales.silver ALTER COLUMN email SET MASK main.security.mask_email;')
print('\n--- production DDL ---\n' + DDL)
assert 'SET ROW FILTER' in DDL and 'SET MASK' in DDL
print('\nModeling judgment: put row filters on the FACT table (tenant_id is the join + liquid-cluster key -> filter prunes files); masks on DIMENSION PII columns. Governed tags drive assignment at scale — tag 20k tables with tenant/PII once, policies follow the tag.')

## Q13 — Data Modeling: Aggregate Facts & Aggregate Navigation [ADVANCED]

**Interviewer (Walmart):** Nightly, hundreds of queries scan the 10B-row base sales fact just to re-derive the same rollups. Design aggregate fact tables and a router that picks the cheapest one — transparently.

**Your task:**
1. **Grain registry:** `agg_monthly_region` (month × region), `agg_daily_region`, `agg_daily_store`, base `fact_sales` (hourly × store × sku) — each with a cost weight
2. **Router:** given requested dims + date grain, pick the CHEAPEST aggregate whose grain rolls up to the request
3. **Consistency contract:** prove rollup(daily) == monthly
4. **Talk track:** when aggregates HURT (freshness lag, storage, extra MERGE pipelines), query-rewrite (optimizer-side) vs app-side routing, and the refresh strategy that keeps them from drifting

In [0]:
# Solution Q13 — grain registry + cheapest-aggregate router + consistency proof
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

AGG_CATALOG = {
    'agg_monthly_region': {'date_grain': 'month', 'dims': {'region'},          'cost': 0.4},
    'agg_daily_region':   {'date_grain': 'day',   'dims': {'region'},           'cost': 10},
    'agg_daily_store':    {'date_grain': 'day',   'dims': {'store'},            'cost': 60},
    'fact_sales':         {'date_grain': 'hour',  'dims': {'store', 'sku'},     'cost': 1000},
}
ORDER = {'hour': 0, 'day': 1, 'month': 2}    # coarser to the right

def navigate(requested_dims, date_grain):
    """Cheapest table whose dims COVER the request and whose date grain is COARSER-OR-EQUAL
    (you can roll an aggregate UP to the request, never down)."""
    candidates = []
    for name, spec in AGG_CATALOG.items():
        if ORDER[spec['date_grain']] > ORDER[date_grain]:      # agg COARSER than request -> cannot roll DOWN to it
            continue
        if requested_dims <= spec['dims'] | {'date'}:          # must cover every requested dim
            candidates.append((spec['cost'], name))
    return sorted(candidates)[0][1] if candidates else None

assert navigate({'region'}, 'month') == 'agg_monthly_region'   # 250x cheaper than base
assert navigate({'region'}, 'day')   == 'agg_daily_region'
assert navigate({'store'}, 'day')   == 'agg_daily_store'
assert navigate({'sku'}, 'day')     == 'fact_sales'           # nothing but base covers sku
print('router picks:', navigate({'region'}, 'month'), '/', navigate({'region'}, 'day'), '/', navigate({'store'}, 'day'), '/', navigate({'sku'}, 'day'))

# consistency contract: rolling the daily agg up MUST equal the monthly agg
daily = spark.createDataFrame([(1, 'R1', 100.0), (2, 'R1', 150.0), (1, 'R2', 50.0)], 'day INT, region STRING, revenue DOUBLE')
monthly = daily.groupBy('region').agg({'revenue': 'SUM'}).withColumnRenamed('SUM(revenue)', 'monthly_rev')
expected = {'R1': 250.0, 'R2': 50.0}
got = {r['region']: r['monthly_rev'] for r in monthly.collect()}
assert got == expected, got
print('rollup consistency:', got, '— the nightly contract check that gates every aggregate refresh.')
print('When aggregates hurt: freshness (yesterday\'s rollup vs live base), storage MERGE pipelines per agg, drift risk. Refresh order: base -> finest agg -> coarser; route BI to aggs, keep base for ad-hoc.')

## Q14 — Snowflake: Clean Rooms [HARD] *(2025–2026 trending)*

**Interviewer (Disney Advertising / ad-tech):** Two brands want joint campaign measurement — "how many users saw BOTH ads and what did they buy?" — but their legal teams say user-level data can NEVER leave either side. Explain and demonstrate the clean-room pattern.

**Cover:**
1. **What a clean room IS:** neutral, governed compute over pre-hashed/encoded data; queries are reviewed policies, not free SQL; neither party sees the other's raw rows
2. **Databricks vs Snowflake implementations:** Databricks clean rooms (Delta Sharing-backed, partner notebooks with restricted code paths) vs Snowflake clean rooms (cleanroom APIs, differential privacy, usage policies)
3. **Demonstrate** a double-blind overlap measurement: both sides submit md5-hashed IDs; the clean room releases ONLY the intersection COUNT
4. **Guardrails:** k-anonymity thresholds (suppress small intersections), output review, join policies, audit logs

In [0]:
# Solution Q14 — double-blind overlap measurement (clean-room semantics, simulated)
import hashlib

# Each side hashes its OWN list before submission — raw emails never cross the boundary
party_a = {hashlib.md5(e.encode()).hexdigest() for e in ['a@x.com', 'b@x.com', 'c@x.com', 'd@x.com']}
party_b = {hashlib.md5(e.encode()).hexdigest() for e in ['c@x.com', 'd@x.com', 'e@y.com', 'f@y.com', 'g@y.com']}

# Inside the clean room: only digests are visible, only the COUNT is released
overlap = party_a & party_b
print('clean-room output: |A ∩ B| =', len(overlap), 'identities — no emails, no digests released')
assert len(overlap) == 2
assert all(len(h) == 32 for h in overlap)         # md5 digests only — nothing reversible leaves

# k-anonymity guardrail: suppress results below the minimum cell size
MIN_CELL = 2
released = len(overlap) if len(overlap) >= MIN_CELL else None
assert released == 2
print(f'k-anonymity gate (k={MIN_CELL}): PASS — intersection released. At k=3 it would be SUPPRESSED.')

TALK_TRACK = {
    'why_clean_rooms': 'Legal said no raw-data sharing; measurement needed joint computation. Neutral compute + hashed inputs + policy-reviewed queries satisfy both.',
    'databricks_flavor': 'Delta Sharing-backed clean rooms: participants keep data in their own accounts; the clean room runs reviewed, restricted notebooks/aggregation queries only.',
    'snowflake_flavor': 'Cleanrooms on Snowflake: provider/consumer APIs, usage policies, differential privacy noise on aggregates.',
    'guardrails': 'k-anonymity suppression, output-size caps, join policy review, full audit trail; differential privacy adds calibrated noise so repeated queries cannot reconstruct the set.',
    'attack_to_mention': 'hash dictionary attacks (md5 of known emailspace) — mitigate with salted/HMAC hashes or private-set-intersection protocols.',
}
for k, v in TALK_TRACK.items():
    print(f'  {k}: {v}')

## Q15 — Snowflake: Horizon Governance Suite [ADVANCED]

**Interviewer (Snowflake Professional Services):** 500 analysts, 20,000 tables, ZERO PII inventory. Explain Snowflake Horizon and build a simulated classification → masking pipeline.

**Cover:**
1. **Horizon components:** data classification (semantic + ML-based), tagging policies, column-level lineage, access requests, security/privacy dashboards, access intelligence
2. **Simulate** the classifier: column names → PII tags (EMAIL / PHONE / FINANCIAL / NAME) via rule + confidence matching
3. **Auto-generate masking policy DDL** per classification
4. **The governance loop:** classification → tags → policies → access intelligence → re-classification; why this scales past manual reviews

In [0]:
# Solution Q15 — simulated Horizon-style classification + auto masking DDL

PII_RULES = {
    'EMAIL':     ['email', 'e_mail'],
    'PHONE':     ['phone', 'mobile', 'msisdn'],
    'SSN':       ['ssn', 'national_id'],
    'NAME':      ['first_name', 'last_name', 'full_name'],
    'FINANCIAL': ['iban', 'account_number', 'card_number'],
}
columns = ['cust_id', 'email_addr', 'first_name', 'phone_number', 'iban', 'order_total']

def classify(col):
    hits = [tag for tag, kws in PII_RULES.items() if any(k in col for k in kws)]
    return hits[0] if hits else None

classified = {c: classify(c) for c in columns}
print('classification results:')
for c, tag in classified.items():
    print(f'  {c:14s} -> {tag or "(no PII signal)"}')
assert classified['email_addr'] == 'EMAIL' and classified['phone_number'] == 'PHONE'
assert classified['iban'] == 'FINANCIAL' and classified['first_name'] == 'NAME'
assert classified['cust_id'] is None and classified['order_total'] is None

# auto-generate masking DDL from the classification (Horizon tagging policies do this declaratively)
MASK_DDL = (
    "CREATE OR REPLACE MASKING POLICY mask_pii AS (val STRING) RETURNS STRING ->\n"
    "  CASE WHEN CURRENT_ROLE() IN ('PII_READER') THEN val ELSE '***MASKED***' END;\n"
    "ALTER TABLE customers MODIFY COLUMN email_addr SET MASKING POLICY mask_pii;\n"
    "-- Horizon equivalent: TAG-BASED — one policy per tag, applied to every column tagged PII_CATEGORY='EMAIL'")
print('\n--- auto-generated masking policy ---\n' + MASK_DDL)
assert 'MASKING POLICY' in MASK_DDL
print('\nGovernance loop: classify (semantic+ML) -> tag -> policy (mask/RAP) -> access intelligence (who could reach this PII?) -> alert + re-classification on schema drift. Manual review cannot keep up with 20k tables; the loop can.')

## Q16 — Monitoring: LLMOps / GenAI Pipeline Observability [HARD] *(2025–2026 trending)*

**Interviewer (Adobe):** A GenAI document-summarization pipeline is in production. Classic freshness/volume monitors don't catch ITS failure modes — models drift, prompts change cost profiles, hallucinations ship silently.

**Telemetry (3 weeks):** cost $14 → $21 → $30; eval score 0.91 → 0.88 → 0.83; p95 latency 3.1 → 3.4 → 3.9s; budget $100/month.

**Your task:**
1. **Cost telemetry:** WoW growth + monthly projection vs budget
2. **Quality telemetry:** golden-set eval drift, hallucination proxy (ungrounded-claim rate), latency regression
3. **Alert design:** projection-based (catch the trajectory BEFORE the month ends), eval-drift paging vs ticket
4. **CI integration:** eval gates on every model/prompt version change — the LLM equivalent of unit tests

In [0]:
# Solution Q16 — LLMOps telemetry: cost trajectory + eval drift + projection alerts

telemetry = [
    # week, pipeline, prompt_toks, completion_toks, cost_usd, p95_latency_s, eval_score
    (41, 'doc_summarizer', 1_200_000, 180_000, 14.0, 3.1, 0.91),
    (42, 'doc_summarizer', 1_900_000, 260_000, 21.0, 3.4, 0.88),
    (43, 'doc_summarizer', 2_600_000, 340_000, 30.0, 3.9, 0.83),
]
costs = [t[4] for t in telemetry]
latencies = [t[5] for t in telemetry]
scores = [t[6] for t in telemetry]

# 1) cost trajectory — WoW growth + monthly projection
wow_growth = [costs[i + 1] / costs[i] - 1 for i in range(len(costs) - 1)]
projected_month = costs[-1] * 4
budget = 100.0
print('WoW cost growth:', [f'{g:.0%}' for g in wow_growth], f'| projected month: ${projected_month:.0f} vs budget ${budget:.0f}')

# 2) quality trajectory — eval drift + latency regression
drift = scores[0] - scores[-1]
latency_growth = latencies[-1] / latencies[0] - 1
print(f'eval drift: -{drift:.2f} over 3 weeks | p95 latency +{latency_growth:.0%}')

# 3) alerts — projection-based (trajectory), not just point-in-time threshold
alerts = []
if projected_month > budget:
    alerts.append('COST: projected $%d > $%d budget — page the pipeline owner NOW, not at month-end' % (projected_month, budget))
if drift > 0.05:
    alerts.append('QUALITY: eval score down %.2f — page; hallucination proxy likely rising with it' % drift)
if latency_growth > 0.2:
    alerts.append('LATENCY: p95 +%.0f%% — ticket: prompt-length creep or model swap side-effect' % (latency_growth * 100))
for a in alerts: print('  ALERT ->', a)
assert all(g > 0.4 for g in wow_growth) and drift > 0.05 and projected_month > budget
assert len(alerts) == 3
print('\nLLMOps checklist to recite: token/cost per pipeline & per tenant, eval-on-golden-set weekly (plus on every prompt/model change — that is the CI gate), groundedness/hallucination rate sampling, cache-hit rate, and a RED dashboard (rate/errors/duration) for the model gateway.')

## Q17 — Monitoring & Orchestration: Kafka+Flink vs Structured Streaming [ADVANCED]

**Interviewer (Uber):** "Should we move our streaming stack to Kafka + Flink?" Give a decision framework, not an opinion.

**Cover:**
1. **Axis by axis:** latency SLO (ms vs seconds), state model (Flink RocksDB local state + incremental checkpoints vs Spark state store), event-time semantics, exactly-once (Flink two-phase-commit sinks vs Spark-to-Delta transactional), recovery time, ops burden (dedicated Flink ops vs Spark team reuse), cost
2. A `choose_engine()` decision function over three real profiles: **ms-latency fraud detection**, **second-latency sessionization**, **lakehouse batch+stream unification**
3. **The hybrid truth:** large platforms run BOTH — Flink owns sub-second stateful serving paths, Structured Streaming owns Lakehouse ETL; explain where the boundary sits and who maintains which

In [0]:
# Solution Q17 — engine decision function + the axis-by-axis matrix

def choose_engine(latency_slo_ms, needs_stateful_windows, needs_exactly_once, team_spark_first):
    if latency_slo_ms <= 100:
        return 'Kafka + Flink (ms-latency, local RocksDB state, incremental checkpoints)'
    if latency_slo_ms <= 5000 and (needs_stateful_windows or not team_spark_first):
        return 'Flink (low-latency stateful, native event-time windows)'
    return 'Spark Structured Streaming (team skill reuse, Lakehouse-native, Delta transactional sink)'

profiles = [
    ('real-time fraud scoring',   (50,   True,  True,  False)),
    ('click sessionization',     (2000, True,  False, True)),
    ('CDC into the lakehouse',   (30000, False, True,  True)),
]
for name, args in profiles:
    print(f'{name:28s} -> {choose_engine(*args)}')
assert 'Flink' in choose_engine(50, True, True, False)
assert 'Flink' in choose_engine(2000, True, False, True)
assert 'Structured Streaming' in choose_engine(30000, False, True, True)

MATRIX = [
    ('Latency SLO',            'ms–100ms',            'sec–min (micro-batches)'),
    ('State',                  'local RocksDB, keyed, large', 'HDFS-backed state store, checkpointed'),
    ('Event time + watermarks','native',               'native (both — Spark needs watermark tuning)'),
    ('Exactly-once',           'two-phase-commit sinks', 'transactional Delta sink + idempotent writes'),
    ('Failure recovery',       'restore from increment checkpoints (secs)', 'replay source from checkpoint (mins)'),
    ('Ops burden',             'dedicated Flink platform team', 'reuses the Spark team + Lakehouse tooling'),
]
print('\naxis'.ljust(26), 'Flink'.ljust(40), 'Structured Streaming')
for axis, flink, sss in MATRIX:
    print(axis.ljust(26), flink.ljust(40), sss)
print('\nHybrid truth: most big platforms run BOTH. Flink owns sub-second stateful serving (fraud, features); Structured Streaming owns lakehouse ETL (CDC, bronze streaming, ML joins). The boundary is the latency SLO + who operates the fleet.')

## Q18 — Monitoring: Reliability Analytics & Auto-Generated Postmortems [ADVANCED]

**Interviewer (Google SRE-style):** You have 10 recent job runs across ingest / model / export with error classes OOM, UPSTREAM_TIMEOUT, SPOT_LOSS. Turn raw run history into DECISIONS.

**Your task:**
1. **MTTR per error class** + frequency — which class deserves the next engineering hour?
2. **Flakiness / fail-rate per job** — the top offender job
3. **Auto-generate the incident timeline** for the worst incident (chronological reconstruction from the run history)
4. **Close the loop:** how these metrics prioritize the reliability backlog (error-budget style: burn rate × user impact)

In [0]:
# Solution Q18 — MTTR / flakiness / auto-timeline from job run history
from collections import defaultdict

runs = [
    # job, run_id, duration_min, status, error_class, hour
    ('ingest',  1, 12, 'FAILED',    'OOM',              3),
    ('ingest',  2,  8, 'SUCCEEDED', None,               4),
    ('ingest',  3, 11, 'FAILED',    'OOM',              3),
    ('ingest',  4,  6, 'SUCCEEDED', None,                5),
    ('model',   5, 40, 'FAILED',    'UPSTREAM_TIMEOUT', 2),
    ('model',   6, 42, 'FAILED',    'UPSTREAM_TIMEOUT', 2),
    ('model',   7, 41, 'FAILED',    'UPSTREAM_TIMEOUT', 2),
    ('model',   8, 45, 'SUCCEEDED', None,                6),
    ('export',  9,  5, 'FAILED',    'SPOT_LOSS',        1),
    ('export', 10,  4, 'SUCCEEDED', None,                1),
]

# 1) MTTR + frequency per error class (duration of failed run = time-to-fix proxy here;
#    in production: detect_time + fix_time from system.lakeflow.job_run_timeline)
by_err = defaultdict(list)
for job, rid, dur, status, err, hour in runs:
    if err:
        by_err[err].append((job, rid, dur, hour))
freq = {e: len(v) for e, v in by_err.items()}
mttr = {e: sum(r[2] for r in v) / len(v) for e, v in by_err.items()}
print('frequency:', freq, '| MTTR:', {k: round(v, 1) for k, v in mttr.items()})
assert freq['UPSTREAM_TIMEOUT'] == 3 and mttr['SPOT_LOSS'] == 5
worst_class = max(freq, key=freq.get)
assert worst_class == 'UPSTREAM_TIMEOUT'

# 2) fail-rate per job -> top offender
by_job = defaultdict(lambda: [0, 0])
for job, rid, dur, status, err, hour in runs:
    by_job[job][1] += 1
    if status == 'FAILED': by_job[job][0] += 1
fail_rate = {j: (f / t, f, t) for j, (f, t) in by_job.items()}
for j, (r, f, t) in fail_rate.items(): print(f'  {j}: {f}/{t} failed = {r:.0%}')
top = max(fail_rate, key=lambda j: fail_rate[j][0])
assert top == 'model'
print('top offender:', top, '— 3 consecutive UPSTREAM_TIMEOUTs at 02:00, not random noise')

# 3) auto-generated timeline for the worst incident (the first OOM streak)
incident = [r for r in runs if r[0] == 'ingest'][:4]
print('\nauto-generated postmortem timeline (ingest incident):')
for job, rid, dur, status, err, hour in sorted(incident, key=lambda r: r[1]):
    print(f'  run {rid:2d} @ {hour:02d}:00 — {status:10s} {err or "-"} ({dur} min)')

# 4) backlog prioritization: burn = frequency × MTTR (user-visible minutes lost)
priority = sorted(((freq[e] * mttr[e], e) for e in freq), reverse=True)
print('\nreliability backlog (minutes lost, desc):', [(e, round(m, 0)) for m, e in priority])
print('-> UPSTREAM_TIMEOUT first (123 lost minutes: add circuit breaker + faster retry), then OOM (23 min: right-size executor memory), SPOT_LOSS is cheap (5 min: keep, autoscaling already absorbs it).')

---
# Day 20 — Score Card

**18 questions · 13 ADVANCED + 5 HARD (Q4, Q8, Q10, Q14, Q16)**

| # | Topic | Difficulty | Solved without hints? (Y/P/N) | Confidence (1–5) | Key gap to review |
|---|-------|:---------:|:------------------------------:|:----------------:|-------------------|
| Q1 | Migration reconciliation with EXCEPT ALL | ADVANCED | | | |
| Q2 | FIFO inventory cost allocation (windows) | ADVANCED | | | |
| Q3 | LFU cache in O(1) | ADVANCED | | | |
| Q4 | Text-to-SQL evaluation harness | HARD | | | |
| Q5 | Leakage-safe stratified split | ADVANCED | | | |
| Q6 | Streaming SCD-dimension enrichment | ADVANCED | | | |
| Q7 | CBO & Delta statistics | ADVANCED | | | |
| Q8 | Platform capacity planning | HARD | | | |
| Q9 | Exactly-once across systems | ADVANCED | | | |
| Q10 | Chaos / DR game day | HARD | | | |
| Q11 | Multi-tenant isolation | ADVANCED | | | |
| Q12 | UC row filters & column masks | ADVANCED | | | |
| Q13 | Aggregate facts & navigation | ADVANCED | | | |
| Q14 | Clean rooms | HARD | | | |
| Q15 | Snowflake Horizon governance | ADVANCED | | | |
| Q16 | LLMOps / GenAI observability | HARD | | | |
| Q17 | Kafka+Flink vs Structured Streaming | ADVANCED | | | |
| Q18 | Reliability analytics & postmortems | ADVANCED | | | |

**Score:** ___ / 90 · **Review tomorrow:** the two lowest-confidence questions first.

> **Day 21 preview (Week 3 finale):** FAANG-level system design — Lakehouse platform design end-to-end, streaming Lakehouse (Kafka/Eventhouse patterns), and the last trending topics before Week 4 begins (day 22+ = FAANG tags).